<img src="../assets/tumor_twin.png" alt="Tumor Twin" width="500"/>

# Coupled PDE demo: HGG patient data + immune–tumor model

This notebook uses the **same patient package and preprocessing as [HGG_Demo.ipynb](HGG_Demo.ipynb)** (`HGG_demo_001`). The spatial model is **`ImmuneResponse3D`** (stacked state `(C, D, H, W)`) instead of **`ReactionDiffusion3D`** (single field `(D, H, W)`).

### Parallel to HGG_Demo

| HGG_Demo | This notebook |
|----------|----------------|
| Step 1 — load data, timeline, imaging, ADC cellularity | [Load patient](#load-patient--adc-derived-cellularity) |
| Step 2 — tumor model + RT/CT | [Treatment specs & immune model](#treatment-specs--immune-model) |
| Steps 3–4 — solver + prediction | [Forward solve](#forward-solve) → [Postprocess (tumor channel)](#postprocess-tumor-channel) |
| Step 5 — scalar QoI + gradients (optional) | [Optional: gradients](#optional-gradients-hgg_demo-step-5) |
| Step 6 — compare to imaging | [Predicted vs measured TCC](#predicted-vs-measured-tcc) |
| Step 7 — calibration residual | [Residuals](#residuals-at-visit-times) → [Calibration (LM)](#calibration-lm-and-visualization) |

**Takeaway:** `solver.solve` returns `(T, C, D, H, W)`. Use `trajectory_to_map_list(trajectory, 0)` (or `extract_trajectory_component` from `tumortwin.models`) for **tumor-only** maps so plots and losses match HGG_Demo.

**Requirements:** Same as HGG_Demo: run from `tutorials/` with `../input_files/HGG_demo_001/`.

---
## Table of contents
- [Optional: Google Colab](#optional-google-colab)
- [Imports & paths](#imports--paths)
- [Load patient & ADC-derived cellularity](#load-patient--adc-derived-cellularity)
- [Treatment specs & immune model](#treatment-specs--immune-model)
- [Forward solve](#forward-solve)
- [Postprocess (tumor channel)](#postprocess-tumor-channel)
- [Predicted vs measured TCC](#predicted-vs-measured-tcc)
- [Optional: gradients (HGG_Demo Step 5)](#optional-gradients-hgg_demo-step-5)
- [Residuals at visit times](#residuals-at-visit-times)
- [Calibration (LM) and visualization](#calibration-lm-and-visualization)
- [Summary](#summary)
---


In [ ]:
# Optional: Google Colab (same pattern as HGG_Demo)
import sys
import importlib.util
from pathlib import Path

if "google.colab" in sys.modules:
    get_ipython().system("pip uninstall -y torchvision torchaudio thinc fastai")
    def is_package_installed(package_name):
        return importlib.util.find_spec(package_name) is not None
    if not is_package_installed("tumortwin"):
        get_ipython().system("pip install git+https://github.com/OncologyModelingGroup/TumorTwin")
    data_path = Path("../input_files/HGG_demo_001")
    if not data_path.exists():
        get_ipython().system("wget https://github.com/OncologyModelingGroup/TumorTwin/raw/refs/heads/main/input_files/HGG_demo_001.tar.gz")
        get_ipython().system("tar -xzvf HGG_demo_001.tar.gz")
        get_ipython().system("mkdir -p ../input_files")
        get_ipython().system("mv HGG_demo_001 ../input_files/HGG_demo_001")


### Imports & paths
Run the Colab cell above if needed. Use the same relative paths as **HGG_Demo** (`../input_files/HGG_demo_001/`).


In [ ]:
# Imports: HGG_Demo stack + ImmuneResponse3D + pde_workflow
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from datetime import timedelta
from pathlib import Path

from pydantic import FilePath

from tumortwin.models import ImmuneResponse3D
from tumortwin.pde_workflow import (
    fields_at_times_from_trajectory,
    initial_pde_state_from_tumor_field,
    select_timepoint_indices,
    spatiotemporal_residual_vector,
    squared_error_loss,
    trajectory_to_map_list,
)
from tumortwin.postprocessing import (
    plot_cellularity_map,
    plot_imaging_summary,
    plot_measured_TCC,
    plot_patient_timeline,
    plot_predicted_TCC,
)
from tumortwin.preprocessing import ADC_to_cellularity, compute_carrying_capacity
from tumortwin.solvers import TorchDiffEqSolver, TorchDiffEqSolverOptions
from tumortwin.types import (
    ChemotherapySpecification,
    CropSettings,
    CropTarget,
    RadiotherapySpecification,
)
from tumortwin.types.hgg_data import HGGPatientData
from tumortwin.utils import daterange, days_since_first

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

%matplotlib inline
matplotlib.rc("font", weight="normal", size=10)
matplotlib.rc("figure", dpi=300)
matplotlib.rc("savefig", dpi=300)


### Load patient & ADC-derived cellularity
**HGG_Demo Step 1:** `HGGPatientData.from_file`, `plot_patient_timeline`, `plot_imaging_summary`, then `ADC_to_cellularity` for every visit.


In [ ]:
# Same paths as HGG_Demo (run this notebook from `tutorials/`)
data_path = Path("../input_files/HGG_demo_001")
PATIENT_INFO_PATH = FilePath(str(data_path / "HGG_demo_001.json"))
IMAGE_PATH = FilePath(str(data_path))
crop_settings = CropSettings(crop_to=CropTarget.ROI_ENHANCE, padding=10, visit_index=-1)

patient_data = HGGPatientData.from_file(
    PATIENT_INFO_PATH, image_dir=IMAGE_PATH, crop_settings=crop_settings
)
patient_data


In [ ]:
plot_patient_timeline(patient_data)
plt.show()
plot_imaging_summary(patient_data)
plt.show()


In [ ]:
measured_cellularity_maps = [
    ADC_to_cellularity(
        visit.adc_image, visit.roi_enhance_image, visit.roi_nonenhance_image
    )
    for visit in patient_data.visits
]
carrying_capacity = compute_carrying_capacity(patient_data.brainmask_image)
print("Number of visits:", len(measured_cellularity_maps), "carrying_capacity:", carrying_capacity)


### Treatment specs & immune model
**HGG_Demo Step 2:** build `RadiotherapySpecification` and `ChemotherapySpecification` from `patient_data`; here the PDE is `ImmuneResponse3D` with `u0 = model.get_initial_state()`.


In [ ]:
# Identical radiotherapy / chemotherapy specifications as HGG_Demo
rt = RadiotherapySpecification(
    alpha=0.025,
    alpha_beta_ratio=10,
    times=[r.time for r in patient_data.radiotherapy],
    doses=[r.dose for r in patient_data.radiotherapy],
)
ct = ChemotherapySpecification(
    sensitivity=0.5,
    decay_rate=9.2420,
    times=[c.time for c in patient_data.chemotherapy],
    doses=[c.dose for c in patient_data.chemotherapy],
)


In [ ]:
# Immune–tumor parameters (order of magnitude similar to HGG k, D); tune for your study
D1 = torch.tensor(0.025, requires_grad=True, device=device)
mu1 = torch.tensor(0.05, requires_grad=True, device=device)
gamma12 = torch.tensor(0.02, requires_grad=True, device=device)
D4 = torch.tensor(0.025, requires_grad=True, device=device)
gamma21 = torch.tensor(0.02, requires_grad=True, device=device)
v = [0.0, 0.0, 0.0]

initial_tumor = torch.from_numpy(measured_cellularity_maps[0].array).float().to(device)

model = ImmuneResponse3D(
    D1=D1,
    mu1=mu1,
    gamma12=gamma12,
    D4=D4,
    gamma21=gamma21,
    v=v,
    patient_data=patient_data,
    initial_time=patient_data.visits[0].time,
    initial_u1=initial_tumor,
    radiotherapy_specification=rt,
    chemotherapy_specifications=[ct],
    require_grad=True,
    device=device,
)

# Stacked IC (2, D, H, W): must match what the solver integrates
u0 = model.get_initial_state()
u_alt = initial_pde_state_from_tumor_field(
    initial_tumor,
    num_components=model.num_state_components,
    other_fill=float(model.u4_source.detach().cpu().item()),
    device=device,
)
torch.testing.assert_close(u0, u_alt)
print("Initial state shape:", tuple(u0.shape))


### Forward solve
**HGG_Demo Steps 3–4:** `TorchDiffEqSolver` + `daterange`; `solve` returns `(t, trajectory)` with shape `(T, C, D, H, W)`.


In [ ]:
solver_options = TorchDiffEqSolverOptions(
    step_size=timedelta(days=0.5),
    use_adjoint=True,
    device=device,
    method="rk4",
)
solver = TorchDiffEqSolver(model, solver_options)


### Postprocess (tumor channel)
**HGG_Demo Step 4 (plots):** convert the trajectory to tumor-only `(D,H,W)` per time via `trajectory_to_map_list(..., 0)`, then TCC and spatial maps as in HGG_Demo.


In [ ]:
timepoints = daterange(
    patient_data.visits[0].time, patient_data.visits[-1].time, timedelta(days=0.5)
)
_, trajectory = solver.solve(timepoints=timepoints, u_initial=u0)
print("Trajectory shape (T, C, D, H, W):", tuple(trajectory.shape))

tumor_maps = trajectory_to_map_list(trajectory, component_idx=0)

fig, ax = plt.subplots(1, 1, figsize=(5, 2.5))
plot_predicted_TCC(tumor_maps, timepoints, ax=ax, carrying_capacity=carrying_capacity)
ax.set_title("Predicted TCC (tumor, component 0)")
plt.tight_layout()
plt.show()

# Spatial comparison at a few visits (HGG_Demo Step 6 style)
visit_slice = patient_data.visit_days[::2]
n = len(visit_slice)
fig, axes = plt.subplots(2, n, figsize=(3 * n, 5), squeeze=False)
t_tensor = torch.tensor(
    [days_since_first(t, timepoints[0]) for t in timepoints],
    dtype=torch.float32,
    device=trajectory.device,
)
idx_vis = select_timepoint_indices(t_tensor, visit_slice, atol=0.51)
for i, vd in enumerate(visit_slice):
    t_idx = idx_vis[i]
    plot_cellularity_map(
        tumor_maps[t_idx].cpu(), patient_data, time=vd, ax=axes[0, i]
    )
    plot_cellularity_map(
        torch.tensor(measured_cellularity_maps[2 * i].array).float(),
        patient_data,
        time=vd,
        ax=axes[1, i],
    )
axes[0, 0].set_ylabel("Predicted (immune)")
axes[1, 0].set_ylabel("Measured (ADC)")
plt.tight_layout()
plt.show()


### Predicted vs measured TCC
**HGG_Demo Step 6:** overlay model TCC on measured TCC from visits (tumor channel only).


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 2.5))
plot_predicted_TCC(tumor_maps, timepoints, ax=ax, carrying_capacity=carrying_capacity)
plot_measured_TCC(
    [m.array for m in measured_cellularity_maps],
    [v.time for v in patient_data.visits],
    ax=ax,
)
ax.legend(["predicted (immune PDE)", "measured (ADC)"])
ax.set_title("TCC: predicted vs measured")
plt.tight_layout()
plt.show()


### Optional: gradients (HGG_Demo Step 5)
Backprop a **scalar** QoI on the **tumor** field only, e.g. final total cell count (same idea as HGG_Demo + `HGG_Gradients.ipynb`).


In [ ]:
from tumortwin.postprocessing import compute_total_cell_count

model.zero_grad()
tcc_final = compute_total_cell_count(tumor_maps[-1], carrying_capacity)
tcc_final.backward()
print("d TCC_final / d D1:", model.D1.grad)
print("d TCC_final / d mu1:", model.mu1.grad)


### Residuals at visit times
**HGG_Demo Step 7:** vector of `pred - meas` over voxels and visits for least squares / LM. Use **tumor** maps only.


In [ ]:
# LM-ready residual on tumor maps at early visits (same spirit as HGG_Demo Step 7)
n_visits_cal = min(4, len(patient_data.visits))
visit_days_cal = patient_data.visit_days[:n_visits_cal]
idx_cal = select_timepoint_indices(t_tensor, visit_days_cal, atol=0.51)
pred_maps = fields_at_times_from_trajectory(trajectory, idx_cal, component_idx=0)
meas_maps = [
    torch.tensor(measured_cellularity_maps[j].array, dtype=torch.float32, device=device)
    for j in range(n_visits_cal)
]
res = spatiotemporal_residual_vector(pred_maps, meas_maps)
loss = squared_error_loss(res)
print("Residual length:", res.numel(), "SSE:", float(loss.detach().cpu()))

# Full spatiotemporal SSE backprop through torchdiffeq is expensive on the full HGG grid.
# For gradients, follow HGG_Demo Step 5 / HGG_Gradients: define a scalar QoI (e.g. TCC at one visit)
# on `tumor_maps[t_idx]` and call `.backward()` on that scalar.


## Summary

- **Data & preprocessing** match **HGG_Demo** (`HGG_demo_001`, ADC, crop, RT/CT).
- **Model:** `ImmuneResponse3D`; state `(2, D, H, W)`; solver output `(T, 2, D, H, W)`.
- **HGG-style plots:** `trajectory_to_map_list(trajectory, 0)` then the same `plot_*` helpers as the scalar tutorial.
- **Calibration:** use `spatiotemporal_residual_vector` + `LMoptimizer` like HGG_Demo Step 7 (wrap a forward pass that returns tumor maps at visit times).

For a **full** step-by-step clone of HGG_Demo (including LM loop and dose sweep) with `ImmuneResponse3D`, add a notebook or copy HGG cells and swap the model — the patterns above are the PDE-specific pieces.
